# Level 2 – Task 2: Wine Quality Prediction

**Internship:** Oasis Infobyte – Data Analytics  
**Objective:** Compare three classifiers for wine quality categories.

## Workflow
1. Download and inspect UCI red wine quality data.
2. Explore class balance and feature distributions.
3. Create low/medium/high quality classes and justify grouping.
4. Split with stratification.
5. Train Random Forest, SGD and SVC.
6. Evaluate accuracy, classification report and confusion matrices.
7. Show Random Forest feature importance and compare models.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")


## 1. Load dataset

The data is the UCI Red Wine Quality dataset. It contains physicochemical measurements and a quality score. The notebook downloads it when run in Colab.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
wine = pd.read_csv(url, sep=";")
display(wine.head())
print("Shape:", wine.shape)
display(wine.info())
display(wine.describe().T)
print("Missing values:")
display(wine.isna().sum())


## 2. Class distribution and EDA

The original score is ordinal and some scores occur much less often than others. This imbalance can make accuracy alone misleading, so we also inspect precision, recall and F1-score.

In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(data=wine, x="quality")
plt.title("Original Wine Quality Score Distribution")
plt.show()

wine.drop(columns="quality").hist(figsize=(14,10), bins=25)
plt.suptitle("Distributions of Physicochemical Features")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11,8))
sns.heatmap(wine.corr(numeric_only=True), cmap="coolwarm", center=0)
plt.title("Wine Feature Correlation Heatmap")
plt.show()


## 3. Feature engineering: Low / Medium / High

For a three-class task, scores are grouped as:
- Low: quality ≤ 5
- Medium: quality = 6
- High: quality ≥ 7

This reduces the number of rare individual score classes and creates an interpretable target. The grouping is a modelling choice, not a universal wine-rating standard.

In [ ]:
def quality_group(q):
    if q <= 5:
        return "Low"
    elif q == 6:
        return "Medium"
    return "High"

wine["quality_group"] = wine["quality"].apply(quality_group)
display(wine["quality_group"].value_counts())
plt.figure(figsize=(6,4))
sns.countplot(data=wine, x="quality_group", order=["Low","Medium","High"])
plt.title("Grouped Quality Class Distribution")
plt.show()


## 4. Prepare features and stratified split

The original numeric quality score is removed from predictors because it directly defines the target group. Stratification preserves class proportions across train and test sets.

In [ ]:
X = wine.drop(columns=["quality", "quality_group"])
y = wine["quality_group"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Train distribution:")
display(y_train.value_counts(normalize=True).rename("proportion"))
print("Test distribution:")
display(y_test.value_counts(normalize=True).rename("proportion"))


## 5. Train three classifiers

Random Forest is tree-based and does not require feature scaling. SGD and SVC are trained with standardized features.

In [ ]:
models = {
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=300, random_state=42,
                                         class_weight="balanced"))
    ]),
    "SGD Classifier": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SGDClassifier(random_state=42, max_iter=2000,
                                class_weight="balanced"))
    ]),
    "SVC": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVC(class_weight="balanced"))
    ])
}

predictions = {}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    rows.append({"Model": name, "Accuracy": accuracy_score(y_test, y_pred)})
    print("\\n", "="*60, "\\n", name)
    print(classification_report(y_test, y_pred, zero_division=0))
display(pd.DataFrame(rows).sort_values("Accuracy", ascending=False))


## 6. Confusion matrices

Confusion matrices show which quality groups are most often confused. This is useful when the classes are imbalanced.

In [ ]:
labels = ["Low", "Medium", "High"]
for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix – {name}")
    plt.show()


## 7. Random Forest feature importance

In [ ]:
rf = models["Random Forest"].named_steps["model"]
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)
display(importance)

plt.figure(figsize=(8,5))
sns.barplot(data=importance, x="Importance", y="Feature")
plt.title("Random Forest Feature Importance")
plt.show()


## 8. Model comparison and conclusion

Use the accuracy table together with each classification report and confusion matrix. Select the model based on balanced performance across classes, not accuracy alone. Note limitations: the dataset is relatively small, quality labels are subjective, and grouped classes simplify the original rating scale.